# KyberPoly2msg-DS — Quick Access Tutorial

Download the `.h5` file from Zenodo (https://doi.org/10.5281/zenodo.22638659) and place it in the same folder as this notebook, or update `DATASET_PATH` below.

In [ ]:
from hashlib import sha3_256

import h5py
import numpy as np

DATASET_PATH = "Kyber_decaps_F439ZI.h5"
BYTE_IDX = 0  # which byte of SHA3-256(message) to use as the label

## 1. Inspect the file structure

In [ ]:
with h5py.File(DATASET_PATH, "r") as f:
    for group in ["profiling", "attack"]:
        for name in ["traces", "message", "k"]:
            ds = f[f"data/{group}/{name}"]
            print(f"data/{group}/{name}: shape={ds.shape} dtype={ds.dtype}")

## 2. Load a slice of profiling data

Labels are constructed offline as `y = SHA3-256(message)[BYTE_IDX]` — this is the value `poly2msg()` reconstructs on-device, not the raw `message` bytes (which are the pre-hash random seed).

In [ ]:
N_PROFILING = 5000

with h5py.File(DATASET_PATH, "r") as f:
    X_profiling = f["data/profiling/traces"][:N_PROFILING].astype(np.float32)
    profiling_message = f["data/profiling/message"][:N_PROFILING]

y_profiling = np.array(
    [sha3_256(bytes(m)).digest()[BYTE_IDX] for m in profiling_message],
    dtype=np.int64,
)
print(X_profiling.shape, y_profiling.shape, "classes:", len(np.unique(y_profiling)))

## 3. Load the attack partition (single fixed key/message)

In [ ]:
with h5py.File(DATASET_PATH, "r") as f:
    n_attack = f["data/attack/traces"].shape[0]           # 50000
    X_attack = f["data/attack/traces"][:n_attack].astype(np.float32)
    attack_message = f["data/attack/message"][:n_attack]  # aligned 1:1 with X_attack

y_attack = np.array(
    [sha3_256(bytes(m)).digest()[BYTE_IDX] for m in attack_message],
    dtype=np.int64,
)
